# DDM Fixed Bound Analysis
## nxx1, seed=42, gain=1.0
Model: v ~ 1 + coherence (fixed threshold)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
%matplotlib inline

In [ ]:
import xarray as xr
idata = xr.open_datatree('/Users/aliciasmacbookair/Desktop/rnn_hssm_output/ddm_fixed_nxx1_s42_g1.0')
print(idata)

## Model Summary

In [ ]:
# Parameter summary
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
summary_data = []
for param in params:
    data = posterior[param].values.flatten()
    summary_data.append({
        'parameter': param,
        'mean': data.mean(),
        'sd': data.std(),
        'hdi_3%': np.percentile(data, 3),
        'hdi_97%': np.percentile(data, 97),
    })
display(pd.DataFrame(summary_data).set_index('parameter').round(3))
print("\nNote: p_outlier fixed at 0.05 (default lapse probability)")

## Traces

In [ ]:
# Plot traces manually
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
fig, axes = plt.subplots(len(params), 2, figsize=(12, 3*len(params)))
for i, param in enumerate(params):
    data = posterior[param].values  # shape: (chains, draws)
    # KDE plot
    for chain in data:
        axes[i, 0].hist(chain, bins=30, alpha=0.5, density=True)
    axes[i, 0].set_title(param)
    axes[i, 0].set_xlabel('value')
    # Trace plot
    for chain in data:
        axes[i, 1].plot(chain, alpha=0.7)
    axes[i, 1].set_title(f'{param} trace')
plt.tight_layout()
plt.show()

## Posterior Distributions

In [ ]:
# Plot posterior distributions
posterior = idata.posterior
params = ['v_Intercept', 'v_coherence', 'a', 'z', 't']
fig, axes = plt.subplots(1, len(params), figsize=(15, 3))
for i, param in enumerate(params):
    data = posterior[param].values.flatten()
    axes[i].hist(data, bins=50, density=True, alpha=0.7, color='steelblue')
    axes[i].axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.3f}')
    axes[i].set_title(param)
    axes[i].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Effect of Coherence on Drift Rate

In [ ]:
import xarray as xr
posterior = idata.posterior
v_intercept = float(posterior['v_Intercept'].mean())
v_coherence = float(posterior['v_coherence'].mean())

coherence_vals = np.linspace(0, 0.15, 100)
drift_rate = v_intercept + coherence_vals * v_coherence

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(coherence_vals, drift_rate, color='blue')
ax.set_xlabel('Coherence (|coh|)')
ax.set_ylabel('Drift rate (v)')
ax.set_title('Effect of coherence on drift rate\nnxx1, seed=42, gain=1.0')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"v_Intercept: {v_intercept:.3f}")
print(f"v_coherence: {v_coherence:.3f}")

## Posterior Predictive Check

In [ ]:
from IPython.display import Image, display
display(Image('/Users/aliciasmacbookair/Desktop/rnn_hssm_output/ddm_fixed_ppc.png'))

## Quantile Probability Plot

In [ ]:
display(Image('/Users/aliciasmacbookair/Desktop/rnn_hssm_output/ddm_fixed_qpp.png'))